# Convolutional Neural Networks

Image classification with PyTorch.

Two problems, one workflow:

- **Binary** - pizza or steak (a Food-101 subset).
- **Multiclass** - 10 food classes.

What we'll cover:

- Turning images into tensors and batches.
- The typical architecture of a CNN.
- A fast end-to-end TinyVGG example.
- The 7-step workflow: data -> preprocess -> baseline -> fit -> evaluate -> improve -> repeat.
- Max pooling, data augmentation and shuffled batches.
- Predicting on your own images, and saving/loading a model.

> Runs locally with `uv run jupyter lab` and in Colab. The setup cell installs the
> one extra package Colab lacks.

## PyTorch image conventions

Three rules cover almost every bug you will meet here:

- Image tensors are **channels first**: `(N, C, H, W)`, not `(N, H, W, C)`.
- Pixel values are floats in `[0, 1]`; `ToTensor()` does the scaling.
- Losses take **raw logits** and integer labels, not probabilities or one-hot vectors.

## Environment setup

In [ ]:
# Colab setup: install notebook-only extras that Colab does not ship with.
import sys

if "google.colab" in sys.modules:
    try:
        import torchinfo  # noqa: F401
    except ModuleNotFoundError:
        import subprocess

        subprocess.run(
            [sys.executable, "-m", "pip", "-q", "install", "torchinfo"],
            check=True,
        )

import os
import random
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchinfo import summary

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Pick the best available device (CUDA / Apple MPS / CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def repo_root(start=None):
    """Walk upwards until we find the folder that holds pyproject.toml."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


DATA_DIR = repo_root() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Image settings shared by every experiment
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
NUM_WORKERS = 2 if device.type == "cuda" else 0

print(f"PyTorch {torch.__version__} | device: {device}")
print(f"Data folder: {DATA_DIR}")

**Takeaways**
- `torchinfo` is the only extra; everything else ships with the base environment.
- `(C, H, W)` is the PyTorch image layout; one extra leading dimension is the batch.
- Set the seed and `model.train()` / `model.eval()` deliberately, every time.
- `EPOCHS` and `BATCH_SIZE` at the top control every experiment; set `EPOCHS = 1` for a quick smoke run.

## Get the data

We use a **pizza vs steak** subset of [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/).
The folder layout is the standard one for image classification:

```
pizza_steak/
├── train/
│   ├── pizza/    750 images
│   └── steak/    750 images
└── test/
    ├── pizza/    250 images
    └── steak/    250 images
```

The download is cached in the shared `data/` folder, so rerunning the cell is cheap.

In [ ]:
PIZZA_STEAK_URL = "https://storage.googleapis.com/ztm_tf_course/food_vision/pizza_steak.zip"


def download_and_extract(url, folder_name, root=DATA_DIR):
    """Download a zip once and extract it into DATA_DIR."""
    target = root / folder_name
    if target.exists():
        print(f"Already available: {target}")
        return target

    zip_path = root / Path(url).name
    if not zip_path.exists():
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, zip_path)
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(root)
    return target


PIZZA_DIR = download_and_extract(PIZZA_STEAK_URL, "pizza_steak")

## Inspect the data (become one with it)

Before modelling, check three things:

1. How many images per split and class.
2. What the class names are (they become the label indices).
3. What a single image actually looks like as an array.

In [ ]:
# Count images in every split/class folder
for dirpath, dirnames, filenames in os.walk(PIZZA_DIR):
    if filenames:
        relative = Path(dirpath).relative_to(DATA_DIR)
        print(f"{len(filenames):4d} images in {relative}")

class_names = sorted(p.name for p in (PIZZA_DIR / "train").iterdir() if p.is_dir())
print("\nClasses:", class_names)

for split in ("train", "test"):
    counts = {
        name: len(list((PIZZA_DIR / split / name).glob("*.jpg")))
        for name in class_names
    }
    print(f"{split:5s}: {counts} | total {sum(counts.values())}")

**Takeaways**
- 1,500 training and 500 test images, balanced across two classes.
- The folder names define the labels: `pizza` -> 0, `steak` -> 1.
- A balanced binary problem has a 50% "random guess" baseline.

In [ ]:
def view_random_image(target_dir, target_class):
    """Plot a random image from target_dir/target_class and return it."""
    folder = Path(target_dir) / target_class
    image_path = random.choice(list(folder.glob("*.jpg")))
    img = Image.open(image_path).convert("RGB")

    plt.imshow(img)
    plt.title(target_class)
    plt.axis(False)
    print(f"{image_path.name} -> {np.array(img).shape} (H, W, C)")
    return img


plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
view_random_image(PIZZA_DIR / "train", "pizza")
plt.subplot(1, 2, 2)
view_random_image(PIZZA_DIR / "train", "steak")
plt.show()

In [ ]:
img = view_random_image(PIZZA_DIR / "train", "steak")

img_array = np.array(img)
print("NumPy shape (H, W, C):", img_array.shape)
print("Pixel range:", img_array.min(), "to", img_array.max())
print("After dividing by 255:", (img_array / 255.0).min(), "to", (img_array / 255.0).max())

**Takeaways**
- An image is just a `(height, width, 3)` array of numbers from `0` to `255`.
- Neural networks train better on small inputs, so pixels are scaled to `[0, 1]` (divide by 255).
- Images come in different sizes; resizing them all to one shape is the first preprocessing step.

## A (typical) architecture of a CNN

| Layer | What it does | PyTorch | Typical value |
| --- | --- | --- | --- |
| Input | Images resized to one shape | `(N, 3, 224, 224)` tensor | 224x224 RGB |
| Convolution | Learns local features (edges, curves, textures) | `nn.Conv2d` | 10-64 filters, 3x3 |
| Activation | Adds non-linearity | `nn.ReLU` | after every convolution |
| Pooling | Keeps the strongest features and halves the size | `nn.MaxPool2d(2)` | 2x2 |
| Flatten | Turns feature maps into one vector | `nn.Flatten` | once |
| Fully connected | Combines features | `nn.Linear` | 1 output (binary) or one per class |
| Output | Raw scores; the loss applies sigmoid/softmax | logits | 1 or N |

### Counting output shapes

For a `(224, 224, 3)` image with 3x3 convolutions, stride 1 and no padding:

- Every `valid` convolution removes **2** pixels: `224 -> 222 -> 220 -> 218`.
- Every `MaxPool2d(2)` **halves** the height and width.
- `nn.Flatten()` turns `(N, C, H, W)` into `(N, C * H * W)`.

PyTorch does not infer the flatten size for `nn.Linear`, so you pass it to
`nn.Linear(in_features=...)` yourself. A wrong number is the most common CNN bug.

## An end-to-end example

The original Food-101 paper reports **50.76%** accuracy with a random forest across
all 101 classes. We use it as a reference baseline: any useful CNN should clear
50% on our easier two-class problem.

The model below is **TinyVGG** from the [CNN Explainer](https://poloclub.github.io/cnn-explainer/):
two convolution blocks with a pooling layer between them, then a linear output.

We jump straight to a working example first, then take the workflow apart step by step.

> Training every model in this notebook takes a while on CPU. Use a GPU runtime
> in Colab, or set `EPOCHS = 1` at the top to confirm the whole notebook runs.

### Preprocess: resize, scale, batch

`transforms.Compose` chains the preprocessing steps:

- `Resize((224, 224))` -> give every image the same shape.
- `ToTensor()` -> rescale to `[0, 1]` **and** switch to `(C, H, W)`.

`datasets.ImageFolder` reads the folder names as classes, and `DataLoader` handles
batching and shuffling.

In [ ]:
# ToTensor: uint8 pixels 0-255 -> float tensor 0-1, and (H, W, C) -> (C, H, W)
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

pizza_train_set = datasets.ImageFolder(PIZZA_DIR / "train", transform=eval_transform)
pizza_test_set = datasets.ImageFolder(PIZZA_DIR / "test", transform=eval_transform)

class_names = pizza_train_set.classes   # index -> name, matches the labels

train_loader = DataLoader(
    pizza_train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    pizza_test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Classes:", class_names)
print("Class -> index:", pizza_train_set.class_to_idx)
print(f"Train images: {len(pizza_train_set)} | Test images: {len(pizza_test_set)}")
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

**Takeaways**
- `ImageFolder` infers classes from subfolder names; labels are integers.
- `ToTensor()` does both jobs you need: scaling and channel-first layout.
- Shuffle the **training** loader; never shuffle the test loader.

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch:", tuple(images.shape), "| dtype:", images.dtype)
print("Pixel range:", images.min().item(), "to", images.max().item())
print("Label batch:", tuple(labels.shape), "| dtype:", labels.dtype)
print("First 10 labels:", labels[:10].tolist())
print("First 10 names: ", [class_names[i] for i in labels[:10]])

**Takeaways**
- One batch is `(32, 3, 224, 224)` and its labels are `(32,)`.
- `ToTensor()` already normalized the pixels, so no manual `/255` is needed.
- The label tensor is `torch.long`; `CrossEntropyLoss` requires that.

### TinyVGG

The flatten size is `10 * 53 * 53`:

```
224 -> 222 -> 220 -> pool 110 -> 108 -> 106 -> pool 53
```

`make_tinyvgg` takes `num_classes` so the same architecture works for binary and
multiclass problems.

In [ ]:
def make_tinyvgg(num_classes=1):
    """TinyVGG from CNN Explainer, for 224x224 RGB images."""
    return nn.Sequential(
        nn.Conv2d(3, 10, kernel_size=3),
        nn.ReLU(),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(10 * 53 * 53, num_classes),
    )


torch.manual_seed(42)
model_1 = make_tinyvgg(num_classes=1).to(device)
summary(model_1, input_size=(1, 3, IMG_SIZE, IMG_SIZE))

In [ ]:
# Training helpers used by every model below
def batch_accuracy(logits, y):
    """Accuracy for binary (1 output) or multiclass (many outputs) logits."""
    if logits.shape[1] == 1:
        return ((logits > 0).float() == y).sum().item()
    return (logits.argmax(dim=1) == y).sum().item()


def evaluate_model(model, loader, loss_fn):
    """Return average loss and accuracy over a DataLoader."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)

            # BCEWithLogitsLoss wants (N, 1) float targets
            if logits.shape[1] == 1:
                labels = labels.float().unsqueeze(1)
            labels = labels.to(device)

            running_loss += loss_fn(logits, labels).item() * images.size(0)
            correct += batch_accuracy(logits, labels)
            total += images.size(0)

    return running_loss / total, correct / total


def train_model(model, loader, loss_fn, optimizer, epochs=EPOCHS, test_loader=None):
    """Train for a number of epochs and record loss and accuracy."""
    history = {"loss": [], "accuracy": []}
    if test_loader is not None:
        history["test_loss"] = []
        history["test_accuracy"] = []

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in loader:
            images = images.to(device)
            logits = model(images)

            if logits.shape[1] == 1:
                labels = labels.float().unsqueeze(1)
            labels = labels.to(device)

            # The five training steps, in order
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += batch_accuracy(logits, labels)
            total += images.size(0)

        history["loss"].append(running_loss / total)
        history["accuracy"].append(correct / total)

        line = (
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"loss {history['loss'][-1]:.4f} | acc {history['accuracy'][-1] * 100:5.1f}%"
        )
        if test_loader is not None:
            test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
            history["test_loss"].append(test_loss)
            history["test_accuracy"].append(test_acc)
            line += f" | test loss {test_loss:.4f} | test acc {test_acc * 100:5.1f}%"
        print(line)

    return history


def plot_loss_curves(history):
    """Plot train/test loss and accuracy side by side."""
    epochs = range(1, len(history["loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["loss"], label="train")
    if "test_loss" in history:
        plt.plot(epochs, history["test_loss"], label="test")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["accuracy"], label="train")
    if "test_accuracy" in history:
        plt.plot(epochs, history["test_accuracy"], label="test")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


def count_parameters(model):
    """Number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()          # sigmoid + binary cross-entropy
optimizer = optim.Adam(model_1.parameters(), lr=0.001)

history_1 = train_model(
    model_1,
    train_loader,
    loss_fn,
    optimizer,
    epochs=EPOCHS,
    test_loader=test_loader,
)

test_loss, test_acc = evaluate_model(model_1, test_loader, loss_fn)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")
print(f"Trainable parameters: {count_parameters(model_1):,}")

In [ ]:
plot_loss_curves(history_1)

**Takeaways**
- Five epochs are enough for this CNN to comfortably beat the 50.76% baseline.
- The model has only ~31k parameters: convolutions reuse the same small filters
  across the whole image, which is why they are parameter-efficient.
- You now own the training loop; every step in it is explicit.

## Using the same model as before (and why it fails)

A plain dense network can accept images too - just flatten them first. That gives
`3 * 224 * 224 = 150,528` inputs per image.

We reuse the small 4-unit network from the classification notebook. Only the data
and the input shape change.

In [ ]:
torch.manual_seed(42)

model_2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3 * IMG_SIZE * IMG_SIZE, 4),
    nn.ReLU(),
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_2.parameters(), lr=0.001)

history_2 = train_model(
    model_2, train_loader, loss_fn, optimizer, epochs=EPOCHS, test_loader=test_loader
)

print(f"\nTrainable parameters: {count_parameters(model_2):,}")
plot_loss_curves(history_2)

**Takeaways**
- ~602k parameters and still stuck near **50%** - the same as guessing.
- Flattening throws away spatial structure: a pixel's neighbours no longer matter.
- More parameters do not automatically mean a better model.

The obvious next move is to add capacity. Let's use three hidden layers of 100.

In [ ]:
torch.manual_seed(42)

model_3 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3 * IMG_SIZE * IMG_SIZE, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1),
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_3.parameters(), lr=0.001)

history_3 = train_model(
    model_3, train_loader, loss_fn, optimizer, epochs=EPOCHS, test_loader=test_loader
)

print(f"\nTrainable parameters: {count_parameters(model_3):,}")
plot_loss_curves(history_3)

**Takeaways**
- ~15,000,000 parameters - roughly 500x the CNN - and still behind `model_1`
  (80.2% vs 85.0%).
- Adding capacity does help a dense network learn, but only very inefficiently.
- This is the core argument for convolutions: better image features with far fewer weights.

## Binary classification: let's break it down

We just went through a lot at once. Here is the workflow, slowly:

1. Become one with the data (visualize).
2. Preprocess the data (resize, scale, batch).
3. Create a model (start with a baseline).
4. Fit the model.
5. Evaluate the model.
6. Adjust parameters and improve.
7. Repeat until satisfied.

### 1. Import and become one with the data

We already inspected the folder counts. Now look at a few examples from the
**training** set after preprocessing - this is exactly what the model sees.

In [ ]:
def plot_random_images(dataset, classes, n=8, seed=42):
    """Plot n random (image, label) pairs from a dataset."""
    random.seed(seed)
    indices = random.sample(range(len(dataset)), n)

    fig, axes = plt.subplots(2, n // 2, figsize=(14, 7))
    for ax, idx in zip(axes.flatten(), indices):
        image, label = dataset[idx]
        ax.imshow(image.permute(1, 2, 0))   # (C, H, W) -> (H, W, C) for plotting
        ax.set_title(classes[label])
        ax.axis(False)

    plt.tight_layout()
    plt.show()


plot_random_images(pizza_train_set, class_names)

**Takeaways**
- Visualize 10-100 samples at the start; it exposes wrong labels, odd crops and class imbalance.
- Pizza images tend to be round and bright; steak images are darker and irregular.
- Those are exactly the kinds of local patterns a convolution can learn.

### 2. Preprocess the data (prepare it for a model)

A **batch** is a small group of images the model looks at before one weight update.

- A whole dataset rarely fits in GPU memory.
- Small batches update the weights many times per epoch, which usually learns faster.
- 32 is a common, safe default.

We already built `train_loader` and `test_loader`. Check what comes out.

In [ ]:
print(f"Train: {len(pizza_train_set)} images -> {len(train_loader)} batches")
print(f"Test:  {len(pizza_test_set)} images -> {len(test_loader)} batches")

images, labels = next(iter(train_loader))
print(f"\nOne batch: images {tuple(images.shape)} | labels {tuple(labels.shape)}")
print("Labels:   ", labels.tolist())
print("Names:    ", [class_names[i] for i in labels])

**Takeaways**
- 1,500 images / 32 per batch = 47 batches per epoch.
- `class_mode="binary"` became a single output column; `0` = pizza, `1` = steak.
- In PyTorch, batching lives in `DataLoader`, not inside the model.

### 3. Create a model (start with a baseline)

Build the simplest CNN that could work: three convolutions, no pooling, one output.
This is intentionally small. Beat it first, then add complexity.

In [ ]:
def make_baseline_cnn(num_classes=1):
    """3 convolutions and no pooling - deliberately simple."""
    return nn.Sequential(
        nn.Conv2d(3, 10, kernel_size=3),
        nn.ReLU(),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.Flatten(),
        nn.Linear(10 * 218 * 218, num_classes),
    )


torch.manual_seed(42)
model_4 = make_baseline_cnn(num_classes=1).to(device)
summary(model_4, input_size=(1, 3, IMG_SIZE, IMG_SIZE))

The `nn.Conv2d` arguments and what they mean:

- **`in_channels` / `out_channels`** - input colour channels (3 for RGB) and how many filters to learn.
- **`kernel_size`** - how big each filter is; `3` means 3x3 pixels. Smaller kernels see finer detail.
- **`stride`** - how far the filter moves each step; default `1`.
- **`padding`** - `0` ("valid") shrinks the output; `padding=1` ("same") keeps it the same size.

A **feature** is any meaningful local pattern. We do not define features by hand;
the filters are learned from data.

**Binary output:** one raw logit, compared with `BCEWithLogitsLoss`. No sigmoid in
the model - the loss applies it internally, which is numerically more stable.

### 4. Fit the model

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_4.parameters(), lr=0.001)

history_4 = train_model(
    model_4, train_loader, loss_fn, optimizer, epochs=EPOCHS, test_loader=test_loader
)

### 5. Evaluate the model

`model.evaluate()` does not exist; `evaluate_model` plays that role. The loss
curves tell you more than one accuracy number.

In [ ]:
test_loss, test_acc = evaluate_model(model_4, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")
print(f"Trainable parameters: {count_parameters(model_4):,}")

plot_loss_curves(history_4)

**Takeaways**
- Train loss keeps falling while test loss flattens and then edges up - that is **overfitting**.
- The ideal curves follow each other closely; a widening gap means memorization.
- One accuracy number hides this. Always plot both curves.

> Here the test split doubles as validation so the curves are comparable. In a
> real project, hold out a separate validation split and keep test untouched.

### 6. Adjust the model parameters

Improving a model usually goes:

0. Create a baseline.
1. Overfit a larger model to prove it can learn.
2. Reduce overfitting.

Ways to reduce overfitting: more data, a simpler model, data augmentation, or
transfer learning. We try the last three ideas in order, starting with **max pooling**.

Pooling keeps the strongest activation in each 2x2 window and throws the rest away.
That halves the spatial size, which cuts parameters and forces the model to keep
only the most useful features.

In [ ]:
def make_pooled_cnn(num_classes=1):
    """Same idea as the baseline, with max pooling after every convolution."""
    return nn.Sequential(
        nn.Conv2d(3, 10, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(10 * 26 * 26, num_classes),
    )


torch.manual_seed(42)
model_5 = make_pooled_cnn(num_classes=1).to(device)
summary(model_5, input_size=(1, 3, IMG_SIZE, IMG_SIZE))

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_5.parameters(), lr=0.001)

history_5 = train_model(
    model_5, train_loader, loss_fn, optimizer, epochs=EPOCHS, test_loader=test_loader
)

test_loss, test_acc = evaluate_model(model_5, test_loader, loss_fn)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")
print(f"Trainable parameters: {count_parameters(model_5):,}")

plot_loss_curves(history_5)

**Takeaways**
- Each `MaxPool2d(2)` halves height and width: `222 -> 111 -> 54 -> 26`.
- Parameters drop from ~477k to ~9k, and test accuracy improves to 86.8%.
- Test loss keeps falling, so this model is not obviously overfitting.
- Fewer features act as a cheap regularizer; augmentation is still worth testing, but it is not required here.

### Data augmentation

**Data augmentation** randomly alters training images (rotate, shift, zoom, flip)
so the model sees more variety. It cannot memorize one exact image, so it learns
more general patterns.

Two rules:

- Augment only the **training** data.
- Augmentation is applied on the fly by the dataset transform; originals stay untouched.

In [ ]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),                              # rotation_range=20
    transforms.RandomAffine(
        degrees=0,
        translate=(0.2, 0.2),                                   # width/height shift
        shear=10,                                               # shear_range=0.2
    ),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),   # zoom_range=0.2
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

sample_path = random.choice(list((PIZZA_DIR / "train" / "pizza").glob("*.jpg")))
original = Image.open(sample_path).convert("RGB")

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(original)
axes[0].set_title("Original")
axes[0].axis(False)

for ax in axes[1:]:
    ax.imshow(train_transform_augmented(original).permute(1, 2, 0))
    ax.set_title("Augmented")
    ax.axis(False)

plt.tight_layout()
plt.show()

In [ ]:
pizza_train_set_aug = datasets.ImageFolder(
    PIZZA_DIR / "train", transform=train_transform_augmented
)
train_loader_aug = DataLoader(
    pizza_train_set_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
print(f"Augmented training images: {len(pizza_train_set_aug)}")

**Takeaways**
- Augmentation makes each epoch harder, so training accuracy usually drops.
- Judge the result by the test score, not the training score.
- It helps most when the model is overfitting; `model_5` was already fitting well.
- Keep transforms realistic for the problem (a vertical food photo may be fine; an upside-down one may not).

In [ ]:
torch.manual_seed(42)
model_6 = make_pooled_cnn(num_classes=1).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_6.parameters(), lr=0.001)

history_6 = train_model(
    model_6,
    train_loader_aug,
    loss_fn,
    optimizer,
    epochs=EPOCHS,
    test_loader=test_loader,
)

test_loss, test_acc = evaluate_model(model_6, test_loader, loss_fn)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")

plot_loss_curves(history_6)

**Takeaways**
- Train accuracy is lower than `model_5` because augmented images are harder to fit.
- Test accuracy also dropped slightly (83.8% vs 86.8%), so augmentation did not help this run.
- Train and test loss are not directly comparable: train loss is measured on augmented images.
- Augmentation is slower because every image is transformed on the fly.

> A common trap: `shuffle=False` makes the loader return one class at a time
> (all pizza, then all steak). Training still runs, but each batch is
> unrepresentative and the curves look broken. Keep `shuffle=True`.

In [ ]:
unshuffled_loader = DataLoader(pizza_train_set, batch_size=8, shuffle=False)

for batch_idx, (_, labels) in enumerate(unshuffled_loader):
    print(f"Batch {batch_idx}: {labels.tolist()}")
    if batch_idx == 2:
        break

**Takeaways**
- `shuffle=False` walks the folders in order: first batch = all pizza, then all steak.
- `shuffle=True` mixes classes, so every update sees both.
- The test loader stays unshuffled only because order does not matter for evaluation.

### 7. Repeat until satisfied

Ideas that usually help next:

- More convolutional layers or more filters.
- Longer training, or a learning-rate sweep.
- More data.
- **Transfer learning** - start from a model already trained on millions of images.

Everything except the last two is **hyperparameter tuning**. Let's return to TinyVGG,
this time with pooling (it already has it) and augmented data.

In [ ]:
torch.manual_seed(42)
model_7 = make_tinyvgg(num_classes=1).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_7.parameters(), lr=0.001)

history_7 = train_model(
    model_7,
    train_loader_aug,
    loss_fn,
    optimizer,
    epochs=EPOCHS,
    test_loader=test_loader,
)

test_loss, test_acc = evaluate_model(model_7, test_loader, loss_fn)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")
print(f"Trainable parameters: {count_parameters(model_7):,}")

plot_loss_curves(history_7)

**Takeaways**
- TinyVGG + augmentation did not beat the simpler pooled model in 5 epochs
  (79.8% vs 86.8%).
- More capacity needs more training; augmentation also slows how fast the model fits.
- Test loss turns up at epoch 5, so training longer without other changes is risky.
- The simplest model that works is often the better baseline.

## Making a prediction with our trained model

To test the model on real, unseen images we need to match the training pipeline
exactly:

1. Open as RGB.
2. Resize to `224x224`.
3. Scale to `[0, 1]` and switch to `(C, H, W)`.
4. Add a batch dimension: `(3, 224, 224)` -> `(1, 3, 224, 224)`.

In [ ]:
CUSTOM_DIR = DATA_DIR / "custom_images"
CUSTOM_DIR.mkdir(parents=True, exist_ok=True)

custom_images = {
    "steak.jpeg": "https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-steak.jpeg",
    "pizza-dad.jpeg": "https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-pizza-dad.jpeg",
    "hamburger.jpeg": "https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-hamburger.jpeg",
    "sushi.jpeg": "https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-sushi.jpeg",
}

for filename, url in custom_images.items():
    path = CUSTOM_DIR / filename
    if not path.exists():
        print("Downloading", filename)
        urllib.request.urlretrieve(url, path)

print("Custom images:", sorted(p.name for p in CUSTOM_DIR.glob("*.jpeg")))

In [ ]:
def load_and_prep_image(filename, img_size=IMG_SIZE):
    """Load an RGB image as a tensor in [0, 1] with shape (3, H, W)."""
    img = Image.open(filename).convert("RGB")
    img = img.resize((img_size, img_size))
    return transforms.ToTensor()(img)


def predict_image(model, filename, classes):
    """Return (predicted class name, confidence) for one image."""
    img = load_and_prep_image(filename)

    model.eval()
    with torch.inference_mode():
        logits = model(img.unsqueeze(0).to(device))   # batch dimension of 1

    if logits.shape[1] == 1:                          # binary: one logit
        probability = torch.sigmoid(logits).item()
        index = int(probability > 0.5)
        confidence = probability if index == 1 else 1 - probability
        return classes[index], confidence

    probabilities = torch.softmax(logits, dim=1).squeeze(0).cpu()   # multiclass
    index = probabilities.argmax().item()
    return classes[index], probabilities[index].item()


def pred_and_plot(model, filename, classes):
    """Predict one image and plot it with the predicted class."""
    predicted_class, confidence = predict_image(model, filename, classes)
    img = load_and_prep_image(filename)

    plt.imshow(img.permute(1, 2, 0))
    plt.title(f"Prediction: {predicted_class} ({confidence * 100:.1f}%)")
    plt.axis(False)
    plt.show()

In [ ]:
steak = load_and_prep_image(CUSTOM_DIR / "steak.jpeg")

print("Image shape (C, H, W):", tuple(steak.shape))
print("Pixel range:", steak.min().item(), "to", steak.max().item())
print("Batch shape:          ", tuple(steak.unsqueeze(0).shape))

model_7.eval()
with torch.inference_mode():
    logits = model_7(steak.unsqueeze(0).to(device))

print("\nRaw logit:", logits.item())
print("Probability of class 1 (steak):", torch.sigmoid(logits).item())

**Takeaways**
- The model always expects a batch, even for one image: use `unsqueeze(0)`.
- The binary model returns **one logit**; `sigmoid` turns it into a probability.
- `> 0.5` maps the probability to a class. The threshold is adjustable.
- Preprocessing must match training exactly, or the predictions are meaningless.

In [ ]:
pred_and_plot(model_7, CUSTOM_DIR / "steak.jpeg", class_names)
pred_and_plot(model_7, CUSTOM_DIR / "pizza-dad.jpeg", class_names)

## Multiclass classification

Same workflow, more classes. Food-101 has 101; we use **10**:

`chicken_curry`, `chicken_wings`, `fried_rice`, `grilled_salmon`, `hamburger`,
`ice_cream`, `pizza`, `ramen`, `steak`, `sushi`.

Only three things change:

1. The output layer gets **10** neurons instead of 1.
2. The loss becomes `nn.CrossEntropyLoss` (softmax + NLL, raw logits in).
3. Predictions take `argmax` over the 10 scores instead of a 0.5 threshold.

### 1. Import and become one with the data

The 10-class subset has 750 training and 250 test images per class, so 7,500 and
2,500 images in total. Expect each epoch to take about five times longer than the
binary problem.

In [ ]:
TEN_FOODS_URL = "https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_all_data.zip"

FOOD_DIR = download_and_extract(TEN_FOODS_URL, "10_food_classes_all_data")

food_train_dir = FOOD_DIR / "train"
food_test_dir = FOOD_DIR / "test"

In [ ]:
food_class_names = sorted(p.name for p in food_train_dir.iterdir() if p.is_dir())

for split in ("train", "test"):
    split_dir = FOOD_DIR / split
    counts = {
        name: len(list((split_dir / name).glob("*.jpg")))
        for name in food_class_names
    }
    print(f"{split:5s}: {sum(counts.values()):5d} images | {counts}")

print("\nClasses:", food_class_names)

**Takeaways**
- 10 balanced classes; folder names give the label order.
- The dataset is 5x larger than pizza/steak, so epochs are slower.
- Always confirm the class list before interpreting predictions.

### 2. Preprocess the data

The pipeline is identical to the binary case: same size, same scaling, same
`ImageFolder`. Only the data and the number of classes differ. Keeping everything
else fixed makes comparisons fair.

In [ ]:
food_train_set = datasets.ImageFolder(food_train_dir, transform=eval_transform)
food_test_set = datasets.ImageFolder(food_test_dir, transform=eval_transform)

food_train_loader = DataLoader(
    food_train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
food_test_loader = DataLoader(
    food_test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Train: {len(food_train_set)} images | Test: {len(food_test_set)} images")
print("Classes:", food_train_set.classes)
print(f"Batches per epoch: {len(food_train_loader)}")

In [ ]:
plot_random_images(food_train_set, food_class_names)

images, labels = next(iter(food_train_loader))
print("One batch:", tuple(images.shape), "| labels:", tuple(labels.shape))
print("Labels:", labels.tolist())

**Takeaways**
- 7,500 / 32 = 235 batches per epoch - five times the binary workload.
- Labels are integers `0-9`, exactly what `CrossEntropyLoss` expects.
- No one-hot encoding is needed anywhere.

### 3. Create a model (start with a baseline)

Reuse TinyVGG, but change the output to `num_classes=10`. No softmax layer:
`CrossEntropyLoss` applies it internally.

In [ ]:
torch.manual_seed(42)
model_8 = make_tinyvgg(num_classes=len(food_class_names)).to(device)

print(f"Trainable parameters: {count_parameters(model_8):,}")
summary(model_8, input_size=(1, 3, IMG_SIZE, IMG_SIZE))

### 4. Fit the model

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_8.parameters(), lr=0.001)

history_8 = train_model(
    model_8,
    food_train_loader,
    loss_fn,
    optimizer,
    epochs=EPOCHS,
    test_loader=food_test_loader,
)

### 5. Evaluate the model

In [ ]:
test_loss, test_acc = evaluate_model(model_8, food_test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")

plot_loss_curves(history_8)

**Takeaways**
- Training accuracy climbs quickly, but the test curves lag far behind - strong overfitting.
- 10 classes are harder than 2; random guessing is now 10%, not 50%.
- The gap, not the absolute score, is the signal that regularization is needed.

### 6. Adjust the model parameters

Ways to fight overfitting, in the order most people try them:

- **Get more data** - not always possible.
- **Simplify the model** - fewer layers or fewer filters.
- **Use data augmentation** - more variety without new files.
- **Use transfer learning** - start from pretrained weights.

Preventing overfitting is called **regularization**.

First, simplify: remove two of the four convolutions.

In [ ]:
def make_simplified_cnn(num_classes):
    """Two convolutions + pooling - less capacity than TinyVGG."""
    return nn.Sequential(
        nn.Conv2d(3, 10, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(10, 10, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(10 * 54 * 54, num_classes),
    )


torch.manual_seed(42)
model_9 = make_simplified_cnn(num_classes=len(food_class_names)).to(device)

print(f"Trainable parameters: {count_parameters(model_9):,}")

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_9.parameters(), lr=0.001)

history_9 = train_model(
    model_9,
    food_train_loader,
    loss_fn,
    optimizer,
    epochs=EPOCHS,
    test_loader=food_test_loader,
)

test_loss, test_acc = evaluate_model(model_9, food_test_loader, loss_fn)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")

plot_loss_curves(history_9)

**Takeaways**
- Removing two convolutions did not reduce parameters (292,800 vs 283,920) because the flattened map is larger.
- Test accuracy improved a little (31.9% vs 26.6%), but the train/test gap is still large.
- Fewer layers alone did not fix overfitting; change the **data** instead.

Now add augmentation, keeping the architecture exactly the same as `model_9`. This
is a controlled experiment: one change at a time.

`model_9` is reused as a blueprint by calling the factory again; a fresh call
gives fresh (untrained) weights.

In [ ]:
food_train_set_aug = datasets.ImageFolder(
    food_train_dir, transform=train_transform_augmented
)
food_train_loader_aug = DataLoader(
    food_train_set_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

torch.manual_seed(42)
model_10 = make_simplified_cnn(num_classes=len(food_class_names)).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_10.parameters(), lr=0.001)

history_10 = train_model(
    model_10,
    food_train_loader_aug,
    loss_fn,
    optimizer,
    epochs=EPOCHS,
    test_loader=food_test_loader,
)

test_loss, test_acc = evaluate_model(model_10, food_test_loader, loss_fn)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")

plot_loss_curves(history_10)

**Takeaways**
- Augmentation closes the train/test gap: test accuracy (24.9%) is even above train.
- Test accuracy is lower than `model_9` (31.9%): five epochs is not enough for augmentation to pay off.
- Augmented epochs are slower because transforms run on the fly.
- Be a serial experimenter: change one variable, keep the rest fixed, compare.

**7. Repeat until satisfied.** More layers, more filters, longer training,
augmentation tweaks, or (the big one) transfer learning. We stop here so the
experiments stay cheap.

## Making a prediction with our trained model

The binary `pred_and_plot` from earlier is already generic: it checks the output
size. With 10 outputs it switches from `sigmoid` + threshold to `softmax` + `argmax`.

Why the switch? The model returns one logit per class. Softmax turns those into
10 probabilities that sum to 1; the largest one wins.

In [ ]:
img = load_and_prep_image(CUSTOM_DIR / "pizza-dad.jpeg")

model_10.eval()
with torch.inference_mode():
    logits = model_10(img.unsqueeze(0).to(device))

probabilities = torch.softmax(logits, dim=1).squeeze(0).cpu()

print("Logits shape:", tuple(logits.shape))
print("Probability sum:", round(probabilities.sum().item(), 3))
print("\nTop 3 classes:")
for probability, index in zip(*probabilities.topk(3)):
    print(f"  {food_class_names[index]}: {probability.item() * 100:.1f}%")

In [ ]:
pred_and_plot(model_10, CUSTOM_DIR / "steak.jpeg", food_class_names)
pred_and_plot(model_10, CUSTOM_DIR / "sushi.jpeg", food_class_names)
pred_and_plot(model_10, CUSTOM_DIR / "pizza-dad.jpeg", food_class_names)
pred_and_plot(model_10, CUSTOM_DIR / "hamburger.jpeg", food_class_names)

**Takeaways**
- Binary: 1 logit -> `sigmoid` -> compare with `0.5`.
- Multiclass: N logits -> `softmax` -> `argmax` over N.
- A confidence score is not a guarantee; many predictions will still be wrong.
- Always keep the class-name order aligned with the training indices.

In [ ]:
model_10.eval()
all_predictions, all_labels = [], []

with torch.inference_mode():
    for images, labels in food_test_loader:
        logits = model_10(images.to(device))
        all_predictions.append(logits.argmax(dim=1).cpu())
        all_labels.append(labels)

y_pred = torch.cat(all_predictions).numpy()
y_true = torch.cat(all_labels).numpy()

print(classification_report(
    y_true,
    y_pred,
    labels=range(len(food_class_names)),
    target_names=food_class_names,
    zero_division=0,
))

plt.figure(figsize=(10, 8))
sns.heatmap(
    confusion_matrix(y_true, y_pred, labels=range(len(food_class_names))),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=food_class_names,
    yticklabels=food_class_names,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix - 10 food classes")
plt.show()

**Takeaways**
- The diagonal shows correct predictions; everything else is a confusion.
- The model collapses toward a few classes: `ice_cream` and `steak` absorb many mistakes.
- `sushi` is almost never predicted (recall 0.00), which a single accuracy number would hide.
- A 5-epoch CNN on 10 classes is a starting point, not a finished product.

## Saving and loading our model

Saving a trained model takes three steps:

1. `torch.save(model.state_dict(), path)` - weights only.
2. Rebuild the same architecture.
3. `model.load_state_dict(torch.load(path))`.

Saving the `state_dict` is smaller and more portable than pickling the whole model.

In [ ]:
SAVE_DIR = repo_root() / "neural-network" / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = SAVE_DIR / "food10_tinyvgg.pth"

torch.save(model_10.state_dict(), MODEL_PATH)
print("Saved weights:", MODEL_PATH)

# Rebuild the architecture, then load the weights into it
loaded_model = make_simplified_cnn(num_classes=len(food_class_names)).to(device)
loaded_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device, weights_only=True)
)
loaded_model.eval()

original_loss, original_acc = evaluate_model(model_10, food_test_loader, loss_fn)
loaded_loss, loaded_acc = evaluate_model(loaded_model, food_test_loader, loss_fn)

print(f"Original: loss {original_loss:.4f} | accuracy {original_acc * 100:.1f}%")
print(f"Loaded:   loss {loaded_loss:.4f} | accuracy {loaded_acc * 100:.1f}%")

**Takeaways**
- Save the `state_dict`, not the model object; loading is explicit.
- The architecture must match exactly before `load_state_dict` works.
- Always call `model.eval()` after loading so dropout/batchnorm behave correctly.
- `map_location=device` lets a GPU-trained file load on a CPU machine.

## Exercises

1. Spend 20 minutes on the [CNN Explainer](https://poloclub.github.io/cnn-explainer/).
   Explain convolution and pooling in your own words.
2. Change TinyVGG's filters from 10 to 32. How does the parameter count and test
   accuracy change?
3. Train the binary model for 10 epochs instead of 5. Plot the curves - when does
   test loss start to rise?
4. Take 10 photos of two different things and build your own binary classifier.
   Which preprocessing step is hardest to get right?
5. Remove `RandomHorizontalFlip` from the augmentation pipeline and retrain.
   Does the result get better or worse? Why might flipping not always be safe?
6. Find a good learning rate for the 10-class problem with a sweep from `1e-5` to `1e-1`.

## Recap: the important bits

| Idea | What to remember |
| --- | --- |
| Images as tensors | `(N, C, H, W)` floats in `[0, 1]` |
| `ImageFolder` | Classes come from subfolder names; labels are integers |
| `DataLoader` | Batching + shuffling; always shuffle the training loader |
| Convolution | Learns local features with few parameters |
| Max pooling | Halves spatial size, keeps the strongest features, cuts parameters |
| Dense on raw pixels | Huge parameter count and poor accuracy - convolutions win |
| Overfitting | Train loss falls while test loss rises; watch the gap |
| Data augmentation | Random transforms on training images only; reduces overfitting |
| Binary output | 1 logit + `BCEWithLogitsLoss`; threshold `sigmoid` at 0.5 |
| Multiclass output | N logits + `CrossEntropyLoss`; take `argmax` |
| Predictions | Match preprocessing, add a batch dimension, use `inference_mode` |
| Saving | Save `state_dict`, rebuild the architecture, `load_state_dict` |
| Workflow | Data -> preprocess -> baseline -> fit -> evaluate -> improve -> repeat |

## Extra resources

- [CNN Explainer](https://poloclub.github.io/cnn-explainer/) - visual, interactive TinyVGG.
- [CS231n: Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/) - the classic deep dive.
- [A guide to convolution arithmetic for deep learning](https://arxiv.org/pdf/1603.07285.pdf) - padding, stride and output sizes.
- [PyTorch vision tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html) - the natural next step: transfer learning.
- [PyTorch image classification tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) - a compact end-to-end CNN example.